# Football Season Intelligence
### Exploratory Data Analysis of Real Football Match Events

Prepared as a portfolio project for football analytics roles. All figures and tables below are computed from the official StatsBomb Open Data repository (https://github.com/statsbomb/open-data); nothing is fabricated or estimated from outside sources.

## Section 1 — Project Overview

### Executive summary
This notebook performs an exploratory analysis of one complete football season using event data from StatsBomb Open Data. The selected competition-season is the **English Premier League 2015/2016**, a full 380-match season in which **Tottenham Hotspur** played all 38 of its matches. Every match file and every event file in the season was downloaded from the public repository and processed (coverage verified in Section 4).

The analysis answers ten football-driven research questions covering goals, team performance, shots, xG, passing, and half-by-half performance, with data-quality checks run before any metric is computed and all findings grounded in the executed results.

### Football questions investigated
1. How are goals distributed across matches?
2. Which teams score the most and concede the fewest goals?
3. Do teams with higher shot volume also create better-quality chances?
4. How does shot location relate to shot outcomes?
5. Which players record the most shots, goals, and expected goals?
6. Which teams accumulate the most passing activity?
7. How do passing completion rates vary between teams?
8. How does performance differ between first and second halves?
9. What is the relationship between expected goals and actual goals?
10. What can — and cannot — be inferred from this dataset?

### Why these questions matter
These are the questions a football analytics department asks every season: how clinical is a team, does shot volume translate into goals, where on the pitch are chances created, which players are the most productive attackers, and how much does passing volume tell us about performance. Each question maps to a concrete, reproducible metric.

### Dataset and selection
Source: StatsBomb Open Data, `data/competitions.json`, official GitHub repository. Selected: **England — Premier League — 2015/2016** (competition_id 2, season_id 27), because it is a complete modern 380-match season with full event coverage and includes Tottenham Hotspur's matches (the user's club of interest).

### Analytical methods
Descriptive exploratory data analysis: match, team, player, shot and pass tables built from raw event records; validation checks for identifiers, coordinates, outcomes and duplicates; statistical plots and mplsoccer pitch maps; correlation and bootstrap uncertainty for home vs away scoring. No predictive modelling is performed (xG modelling is deferred to a later project).

### Tools used
Python 3.11, pandas, NumPy, Matplotlib, Seaborn, mplsoccer, requests, Jupyter/nbclient. Reusable code lives in `src/` (data_loader, data_validation, football_metrics, visualizations).

### Scope and limitations
- Only one season (380 matches) is analysed; results describe that season and do not generalise.
- Event data records actions, not player tracking; tactical positioning off the ball cannot be observed.
- xG values come from StatsBomb (`statsbomb_xg`); no xG model is trained here.
- Minutes played are derived from lineup position intervals where available; per-90 rates are not used as headline metrics.

### What the reader will learn
How goals were distributed in Premier League 2015/2016, how teams compare on goals, shots, xG and passing, where shots are taken from and with what outcome, which attackers were most productive, how play differs between halves, and how closely xG tracked actual goals — together with the caveats that apply to each conclusion.

## Section 2 — Research Questions

Each question lists the metric used and how it should be interpreted. Where a metric cannot be calculated reliably, it is marked unavailable rather than substituted.

### Q1. How are goals distributed across matches?
**Metric:** total goals per match (`home_goals + away_goals`), shown as a frequency distribution. **Interpretation:** the typical match shape and how common high-scoring games are in this season.

### Q2. Which teams score the most and concede the fewest goals?
**Metric:** goals for and goals against per team from match scores; goal difference. **Interpretation:** attacking and defensive output over the season (raw totals, not adjusted for opponent strength).

### Q3. Do teams with higher shot volume also create better-quality chances?
**Metric:** shots per team vs goals per team; shots vs xG per team. **Interpretation:** volume versus quality of chance creation.

### Q4. How does shot location relate to shot outcomes?
**Metric:** shot map of shot locations, coloured by outcome (goal vs non-goal), plus xG distribution. **Interpretation:** where on the pitch goals are scored from (StatsBomb pitch, 120×80, attacking left → right).

### Q5. Which players record the most shots, goals, and expected goals?
**Metric:** per-player shots, goals, xG from shot events (minutes played shown where derivable from lineups). **Interpretation:** raw attacker productivity, not a complete quality measure.

### Q6. Which teams accumulate the most passing activity?
**Metric:** passes attempted per team. **Interpretation:** passing volume only — not a measure of tactical quality.

### Q7. How do passing completion rates vary between teams?
**Metric:** pass completion % (completed passes / attempted passes; see definition in Section 7). **Interpretation:** reliability in possession, not superiority by itself.

### Q8. How does performance differ between first and second halves?
**Metric:** goals scored in period 1 vs period 2, from shot events. **Interpretation:** whether scoring is concentrated in either half.

### Q9. What is the relationship between expected goals and actual goals?
**Metric:** team xG vs actual goals; shot-level xG distribution. **Interpretation:** whether teams over- or under-performed their expected goal totals (small-sample caution applies).

## Section 3 — Environment and Imports

Libraries are imported cleanly, a fixed random seed is set for the bootstrap resampling in Section 14, and a consistent, restrained plotting theme is applied via `src/visualizations.py`. Key package versions are printed.

### Q10. What can be concluded, and what cannot?
**Metric:** n/a — synthesis. **Interpretation:** descriptive findings only; no causal claims, no statistical significance claims without support, no extrapolation beyond this season.

## Section 4 — Data Acquisition

The official StatsBomb competition index is loaded, the Premier League 2015/2016 is selected from actual availability, match metadata and every event file are downloaded with local caching, and coverage is verified before any metric is computed.

In [1]:
import warnings
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from IPython.display import display, HTML

# Make src/ importable no matter where the notebook is launched from.
# Resolve the project root from the current working directory, its parents,
# or sys.path[0] (Jupyter kernels often use the notebook directory there).
_PROJECT_ROOT = None
for _cand in [Path.cwd()] + list(Path.cwd().parents) + ([Path(sys.path[0])] if sys.path else []):
    try:
        if _cand.is_dir() and (_cand / "requirements.txt").exists():
            _PROJECT_ROOT = _cand
            break
    except (OSError, RuntimeError):
        continue
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root (requirements.txt)."
        " cwd: " + str(Path.cwd()) +
        " sys.path[0]: " + str(sys.path[0] if sys.path else None) +
        " Run this notebook from the football-season-intelligence folder."
    )
PROJECT_ROOT = _PROJECT_ROOT
print("Notebook cwd: " + str(Path.cwd()))
print("Project root: " + str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import (
    BASE_URL, load_competitions, find_competition_season, load_matches,
    load_season_events, PITCH_LENGTH, PITCH_WIDTH,
)
from data_validation import (
    validate_match_metadata, validate_events, validate_shots, validate_passes,
    validate_season_coverage, summarize_validation,
)
from football_metrics import (
    build_match_table, build_shot_table, build_pass_table, build_team_table,
    build_player_table, minutes_from_lineups,
)
from visualizations import setup_style, save_fig

setup_style()
warnings.filterwarnings("always")
np.random.seed(42)  # fixed seed for any resampling

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("seaborn:", sns.__version__)
print("mplsoccer:", __import__("mplsoccer").__version__)
print("requests:", requests.__version__)
print("StatsBomb pitch: %.0f x %.0f (attacking left -> right, goal at x = %.0f)" % (PITCH_LENGTH, PITCH_WIDTH, PITCH_LENGTH))

Notebook cwd: D:\Projects\Football\Football season intelligence\football-season-intelligence
Project root: D:\Projects\Football\Football season intelligence\football-season-intelligence


Project root: D:\Projects\Football\Football season intelligence\football-season-intelligence
Python: 3.11.9
pandas: 3.0.6
numpy: 2.4.6
matplotlib: 3.11.2
seaborn: 0.13.2
mplsoccer: 1.8.1
requests: 2.34.2
StatsBomb pitch: 120 x 80 (attacking left -> right, goal at x = 120)


In [2]:
CACHE_DIR = PROJECT_ROOT / "data" / "raw"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Loading competitions index from StatsBomb Open Data ...")
competitions = load_competitions(CACHE_DIR)
print("Loaded " + str(len(competitions)) + " competition-season records.")

TARGET_COMPETITION = "Premier League"
TARGET_SEASON = "2015/2016"

try:
    selection = find_competition_season(competitions, TARGET_COMPETITION, TARGET_SEASON)
except Exception as exc:
    raise SystemExit(
        "Could not select " + TARGET_COMPETITION + " " + TARGET_SEASON +
        ": " + str(exc) + "\n"
        "Download the index manually from " + BASE_URL + "/competitions.json"
    )

COMPETITION_ID = selection["competition_id"]
SEASON_ID = selection["season_id"]
print(
    "Selected: " + selection["country_name"] + " — " +
    selection["competition_name"] + " " + selection["season_name"] +
    " (competition_id=" + str(COMPETITION_ID) + ", season_id=" + str(SEASON_ID) + ")"
)
print("match_available flag: " + str(selection.get("match_available")))

print("Loading match metadata for " + TARGET_COMPETITION + " " + TARGET_SEASON + " ...")
matches = load_matches(COMPETITION_ID, SEASON_ID, CACHE_DIR)
print("Loaded " + str(len(matches)) + " match records.")

dates = [m.get("match_date") for m in matches if m.get("match_date")]
if dates:
    print("Season dates: " + min(dates) + " to " + max(dates))
else:
    print("No dates found.")
teams = set()
for m in matches:
    teams.add((m.get("home_team") or {}).get("home_team_name"))
    teams.add((m.get("away_team") or {}).get("away_team_name"))
tottenham_present = "Tottenham Hotspur" in teams
print(
    "Teams in season: " + str(len(teams)) +
    "; Tottenham Hotspur present: " + str(tottenham_present)
)

Loading competitions index from StatsBomb Open Data ...
Loaded 80 competition-season records.
Selected: England — Premier League 2015/2016 (competition_id=2, season_id=27)
match_available flag: 2025-12-17T14:38:00.447595
Loading match metadata for Premier League 2015/2016 ...
Loaded 380 match records.
Season dates: 2015-08-08 to 2016-05-17
Teams in season: 20; Tottenham Hotspur present: True


## Section 5 — Data Understanding

Match count, event count, teams and players, event-type distribution, metadata, attributes, missing values, duplicates and data types are inspected.

In [3]:
print("Loading event files for all " + str(len(matches)) +
      " matches (cached under data/raw/events/)...")
events_by_match = load_season_events(
    [m["match_id"] for m in matches], CACHE_DIR, verbose=False
)
loaded = len(events_by_match)
print()
print("Matches with events loaded: " + str(loaded) + " / " + str(len(matches)))
print("All matches processed: " + str(loaded == len(matches)))
if loaded != len(matches):
    missing = sorted(set(m["match_id"] for m in matches) - set(events_by_match))
    print("Missing match ids: " + str(missing))

Loading event files for all 380 matches (cached under data/raw/events/)...



Matches with events loaded: 380 / 380
All matches processed: True


In [4]:
n_events = sum(len(ev) for ev in events_by_match.values())
print("Total events loaded: " + format(n_events, ","))

# Event type distribution across the season
event_type_counts = {}
for evs in events_by_match.values():
    for e in evs:
        t = e.get("type", {})
        name = t.get("name") if isinstance(t, dict) else str(t)
        event_type_counts[name] = event_type_counts.get(name, 0) + 1
event_df = (
    pd.DataFrame(list(event_type_counts.items()), columns=["event_type", "count"])
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print("Event-type distribution:")
display(event_df.head(15))

# Teams and players
teams_seen = set()
players_seen = set()
for evs in events_by_match.values():
    for e in evs:
        team = e.get("team")
        if isinstance(team, dict):
            teams_seen.add(team.get("name"))
        player = e.get("player")
        if isinstance(player, dict) and player.get("id") is not None:
            players_seen.add(player["id"])
print("Unique teams appearing in events: " + str(len(teams_seen)))
print("Unique players appearing in events: " + str(len(players_seen)))

Total events loaded: 1,313,773


Event-type distribution:


,event_type,count
0,Pass,368619
1,Ball Receipt*,340314
2,Carry,276949
3,Pressure,115402
4,Ball Recovery,40943
5,Duel,32290
6,Clearance,21645
7,Block,14839
8,Dribble,13721
9,Goal Keeper,11777


Unique teams appearing in events: 20
Unique players appearing in events: 549


## Section 6 — Data Quality and Validation

Validation runs before any metric: match metadata, per-match event streams (identifiers, teams, players, coordinates, duplicates), shot and pass outcomes, and season coverage cross-checks.

In [5]:
from collections import Counter

# Duplicate event ids within matches
dup_check = []
for mid, evs in events_by_match.items():
    ids = [e.get("id") for e in evs if e.get("id")]
    dupes = [i for i, c in Counter(ids).items() if c > 1]
    if dupes:
        dup_check.append((mid, dupes[:3]))
print("Matches with duplicate event ids: " + str(len(dup_check)))
if dup_check:
    print(dup_check[:5])

# Sample records: one match, first few events
sample_mid = sorted(events_by_match)[0]
print("\nSample events from match " + str(sample_mid) + ":")
display(pd.DataFrame(events_by_match[sample_mid])[[
    "id", "index", "period", "minute", "second", "type", "team", "player", "location"
]].head(8))

Matches with duplicate event ids: 0

Sample events from match 3753972:


,id,index,period,minute,second,type,team,player,location
0,6bbafb9c-9e4f-45bf-80f2-6f93c8b4d5e2,1,1,0,0,"{'id': 35, 'name': 'Starting XI'}","{'id': 26, 'name': 'Swansea City'}",NaN,NaN
1,07aa1c9f-aff7-42aa-b763-39a478b8e5ea,2,1,0,0,"{'id': 35, 'name': 'Starting XI'}","{'id': 28, 'name': 'AFC Bournemouth'}",NaN,NaN
2,a893074f-957f-4272-8d64-92020f98337f,3,1,0,0,"{'id': 18, 'name': 'Half Start'}","{'id': 28, 'name': 'AFC Bournemouth'}",NaN,NaN
3,0ee3b84e-9975-48e0-819a-daeee9dc8880,4,1,0,0,"{'id': 18, 'name': 'Half Start'}","{'id': 26, 'name': 'Swansea City'}",NaN,NaN
4,b1c7e9af-19b2-47d8-855f-14aa3209648c,5,1,0,0,"{'id': 30, 'name': 'Pass'}","{'id': 26, 'name': 'Swansea City'}","{'id': 15996, 'name': 'Éderzito António Macedo...","[61.0, 40.1]"
5,e7f3536d-3209-4ed2-a3a4-cd1519e44f7f,6,1,0,0,"{'id': 42, 'name': 'Ball Receipt*'}","{'id': 26, 'name': 'Swansea City'}","{'id': 3057, 'name': 'Jonjo Shelvey'}","[60.2, 40.7]"
6,575d8b5a-1b10-4f34-98df-579cc0a52a2e,7,1,0,0,"{'id': 30, 'name': 'Pass'}","{'id': 26, 'name': 'Swansea City'}","{'id': 3057, 'name': 'Jonjo Shelvey'}","[60.5, 42.1]"
7,70fc4868-b11b-4f0d-adc7-1e819aa8abe9,8,1,0,1,"{'id': 17, 'name': 'Pressure'}","{'id': 28, 'name': 'AFC Bournemouth'}","{'id': 3346, 'name': 'Joshua King'}","[64.3, 41.6]"


## Section 7 — Build Reusable Analytical Tables

Match, team, player and shot tables are built from the events with explicit definitions (goals from shot outcomes, shots on target, pass completion, xG), cross-validated against the official scoreboard.

In [6]:
print("Section 6 — Data quality and validation")
print("=" * 60)

match_report = validate_match_metadata(matches)
print("\nMatch metadata: " + ("OK" if match_report["valid"] else "ISSUES"))
print("  matches: " + str(match_report["n_matches"]) +
      ", unique ids: " + str(match_report["n_unique_matches"]))
for w in match_report.get("warnings", []):
    print("  warning: " + w)
for err in match_report.get("errors", []):
    print("  ERROR: " + err)

assert match_report["valid"], "Match metadata failed validation"
assert match_report["n_unique_matches"] == match_report["n_matches"]

# Validate each match's events (summarise, do not print every match)
all_warnings = []
all_errors = []
n_shots = n_passes = 0
for mid, evs in events_by_match.items():
    rep = validate_events(evs, mid)
    if not rep["valid"]:
        all_errors.append((mid, rep["errors"]))
    all_warnings.extend([(mid, w) for w in rep.get("warnings", [])])
    n_shots += rep["event_types"].get("Shot", 0)
    n_passes += rep["event_types"].get("Pass", 0)

print("\nEvent validation: " + str(len(events_by_match)) + " match streams checked")
print("  total shots: " + format(n_shots, ",") + " | total passes: " + format(n_passes, ","))
print("  matches with validation errors: " + str(len(all_errors)))
if all_warnings:
    print("  non-fatal warnings (sample):")
    for mid, w in all_warnings[:8]:
        print("    match " + str(mid) + ": " + w)
assert not all_errors, "Event validation errors found"

# Shot and pass validation on the pooled data
all_shots = [e for evs in events_by_match.values() for e in evs if e.get("type", {}).get("name") == "Shot"]
all_passes = [e for evs in events_by_match.values() for e in evs if e.get("type", {}).get("name") == "Pass"]
shot_rep = validate_shots(all_shots)
pass_rep = validate_passes(all_passes)
print("\nShot validation: " + str(shot_rep["n_shots"]) + " shots")
print("  outcomes: " + str(dict(sorted(shot_rep["outcomes"].items(), key=lambda kv: str(kv[0])))))
print("  shots without xG: " + str(shot_rep["shots_without_xg"]))
if "xg_sum" in shot_rep:
    print("  xG range: %.4f to %.4f; season xG sum: %.2f" % (
        shot_rep["xg_min"], shot_rep["xg_max"], shot_rep["xg_sum"]))
print("\nPass validation: " + str(pass_rep["n_passes"]) + " passes")
print("  outcomes: " + str(dict(sorted(pass_rep["outcomes"].items(), key=lambda kv: str(kv[0])))))
print("  passes without outcome tag: " + str(pass_rep["passes_without_outcome"]))
print("  passes with 'Unknown' outcome: " + str(pass_rep["unknown_outcome_passes"]))

# Season coverage cross-check
cov = validate_season_coverage(matches, events_by_match)
print("\nSeason coverage: expected " + str(cov["expected_matches"]) +
      " matches, events loaded for " + str(cov["matches_with_events"]) +
      "; complete = " + str(cov["complete_coverage"]))
assert cov["complete_coverage"], "Not all matches have event files"
print("\nAll validation checks passed.")

Section 6 — Data quality and validation

Match metadata: OK
  matches: 380, unique ids: 380



Event validation: 380 match streams checked
  total shots: 9,908 | total passes: 368,619
  matches with validation errors: 0
  non-fatal warnings (sample):
    match 3754217: 17 events missing a usable player id
    match 3754117: 15 events missing a usable player id
    match 3754296: 11 events missing a usable player id
    match 3753983: 11 events missing a usable player id
    match 3754160: 14 events missing a usable player id
    match 3754129: 15 events missing a usable player id
    match 3754309: 17 events missing a usable player id
    match 3754047: 14 events missing a usable player id



Shot validation: 9908 shots
  outcomes: {'Blocked': 2880, 'Goal': 988, 'Off T': 3197, 'Post': 170, 'Saved': 2209, 'Saved Off Target': 45, 'Saved to Post': 23, 'Wayward': 396}
  shots without xG: 0
  xG range: 0.0002 to 0.9951; season xG sum: 971.38

Pass validation: 368619 passes
  outcomes: {'Incomplete': 75035, 'Injury Clearance': 360, None: 282379, 'Out': 7503, 'Pass Offside': 1381, 'Unknown': 1961}
  passes without outcome tag: 282379
  passes with 'Unknown' outcome: 1961

Season coverage: expected 380 matches, events loaded for 380; complete = True

All validation checks passed.


In [7]:
match_table = build_match_table(matches)
shot_table = build_shot_table(events_by_match)
pass_table = build_pass_table(events_by_match)
team_table = build_team_table(match_table, shot_table, pass_table)

# Cross-validate shot-derived goals vs authoritative match scores
score_compare = match_table.copy()
shot_goals = (
    shot_table[shot_table["is_goal"]]
    .groupby(["match_id", "team"])
    .size()
    .reset_index(name="shot_goals")
)

def shot_goals_for(row):
    sub = shot_goals[shot_goals["match_id"] == row["match_id"]]
    home_goals = int(sub[sub["team"] == row["home_team"]]["shot_goals"].sum()) if len(sub) else 0
    away_goals = int(sub[sub["team"] == row["away_team"]]["shot_goals"].sum()) if len(sub) else 0
    # goals not attributed to either team (should not happen in a well-formed export)
    other = int(sub[~sub["team"].isin([row["home_team"], row["away_team"]])]["shot_goals"].sum())
    return pd.Series(
        {"home_shot_goals": home_goals, "away_shot_goals": away_goals, "other_shot_goals": other}
    )

cross = match_table.apply(shot_goals_for, axis=1)
score_compare = pd.concat([match_table.reset_index(drop=True), cross], axis=1)
score_compare["home_diff"] = score_compare["home_goals"] - score_compare["home_shot_goals"]
score_compare["away_diff"] = score_compare["away_goals"] - score_compare["away_shot_goals"]
mismatches = score_compare[(score_compare["home_diff"] != 0) | (score_compare["away_diff"] != 0)]
print("Matches where shot-derived goals differ from scoreboard: " +
      str(len(mismatches)) + " / " + str(len(score_compare)))
if not mismatches.empty:
    print(mismatches[
        ['match_id','date','home_team','away_team','home_goals','away_goals',
         'other_shot_goals']
    ].head(10).to_string(index=False))
print()
print("Match table (first 10 rows):")
display(match_table.head(10))
print("\nShot table shape: " + str(shot_table.shape))
print("Pass table shape: " + str(pass_table.shape))

Matches where shot-derived goals differ from scoreboard: 38 / 380
 match_id       date         home_team         away_team  home_goals  away_goals  other_shot_goals
  3754217 2015-09-19           Chelsea           Arsenal           2           0                 0
  3754160 2015-12-05           Arsenal        Sunderland           3           1                 0
  3754041 2016-05-15           Arsenal       Aston Villa           4           0                 0
  3753978 2015-09-26       Southampton      Swansea City           3           1                 0
  3754074 2015-10-04      Swansea City Tottenham Hotspur           2           2                 0
  3754082 2015-11-01           Everton        Sunderland           6           2                 0
  3754097 2015-08-08 Manchester United Tottenham Hotspur           1           0                 0
  3754115 2015-10-17           Chelsea       Aston Villa           2           0                 0
  3754185 2015-11-21           Watford Manc

,match_id,date,kick_off,home_team,away_team,home_goals,away_goals,total_goals,outcome_home,match_week,competition_stage,stadium,referee
0,3754217,2015-09-19,11:45:00.000,Chelsea,Arsenal,2,0,2,W,6,"{'id': 1, 'name': 'Regular Season'}","{'id': 10, 'name': 'Stamford Bridge', 'country...","{'id': 6, 'name': 'Mike Dean', 'country': {'id..."
1,3754117,2015-12-13,12:30:00.000,Aston Villa,Arsenal,0,2,2,L,16,"{'id': 1, 'name': 'Regular Season'}","{'id': 211, 'name': 'Villa Park', 'country': {...","{'id': 12, 'name': 'Kevin Friend', 'country': ..."
2,3754296,2015-12-21,19:00:00.000,Arsenal,Manchester City,2,1,3,W,17,"{'id': 1, 'name': 'Regular Season'}","{'id': 3, 'name': 'Emirates Stadium', 'country...","{'id': 5, 'name': 'Andre Marriner', 'country':..."
3,3753983,2015-10-31,14:00:00.000,Swansea City,Arsenal,0,3,3,L,11,"{'id': 1, 'name': 'Regular Season'}","{'id': 45, 'name': 'Swansea.com Stadium', 'cou...","{'id': 12, 'name': 'Kevin Friend', 'country': ..."
4,3754160,2015-12-05,14:00:00.000,Arsenal,Sunderland,3,1,4,W,15,"{'id': 1, 'name': 'Regular Season'}","{'id': 3, 'name': 'Emirates Stadium', 'country...","{'id': 9, 'name': 'Robert Madley', 'country': ..."
5,3754129,2015-08-24,19:00:00.000,Arsenal,Liverpool,0,0,0,D,3,"{'id': 1, 'name': 'Regular Season'}","{'id': 3, 'name': 'Emirates Stadium', 'country...","{'id': 1, 'name': 'Michael Oliver', 'country':..."
6,3754309,2015-12-26,18:45:00.000,Southampton,Arsenal,4,0,4,W,18,"{'id': 1, 'name': 'Regular Season'}","{'id': 4591, 'name': 'St. Mary's Stadium', 'co...","{'id': 3, 'name': 'Jonathan Moss', 'country': ..."
7,3754047,2015-11-29,15:15:00.000,Liverpool,Swansea City,1,0,1,W,14,"{'id': 1, 'name': 'Regular Season'}","{'id': 6, 'name': 'Anfield ', 'country': {'id'...","{'id': 8, 'name': 'Anthony Taylor', 'country':..."
8,3754058,2016-01-02,16:00:00.000,Leicester City,AFC Bournemouth,0,0,0,D,20,"{'id': 1, 'name': 'Regular Season'}","{'id': 20, 'name': 'King Power Stadium ', 'cou...","{'id': 5, 'name': 'Andre Marriner', 'country':..."
9,3754245,2015-10-17,16:00:00.000,West Bromwich Albion,Sunderland,1,0,1,W,9,"{'id': 1, 'name': 'Regular Season'}","{'id': 33, 'name': 'The Hawthorns', 'country':...","{'id': 4, 'name': 'Martin Atkinson', 'country'..."



Shot table shape: (9908, 18)
Pass table shape: (368619, 14)


In [8]:
print("Team-level table (ranked by points):")
display(team_table)
print("\nTeams: " + str(len(team_table)))

Team-level table (ranked by points):


,rank,team,matches_played,wins,draws,losses,points,goals_for,goals_against,goal_difference,...,shot_accuracy_pct,xg_for,xg_against,xg_difference,shots_per_match,goals_per_match,passes_attempted,passes_completed,pass_completion_pct,passes_per_match
0,1,Leicester City,38,23,12,3,81,67,36,31,...,34.095238,66.28,46.13,20.15,13.8,1.76,15093,10509,69.6,397.2
1,2,Arsenal,38,20,11,7,71,62,33,29,...,35.506003,65.09,33.57,31.51,15.3,1.63,22709,18886,83.2,597.6
2,3,Tottenham Hotspur,38,19,13,6,70,68,32,36,...,37.347561,62.36,35.56,26.79,17.3,1.79,20624,16341,79.2,542.7
3,4,Manchester United,38,19,9,10,66,46,33,13,...,32.482599,42.20,38.52,3.68,11.3,1.21,21890,17744,81.1,576.1
4,5,Manchester City,38,19,9,10,66,71,39,32,...,33.876221,64.56,39.06,25.50,16.2,1.87,22340,18263,81.8,587.9
5,6,Southampton,38,18,9,11,63,57,41,16,...,31.688805,54.22,39.75,14.47,13.9,1.50,17995,13660,75.9,473.6
6,7,West Ham United,38,16,14,8,62,64,49,15,...,31.842576,54.53,54.27,0.26,14.7,1.68,16773,12794,76.3,441.4
7,8,Liverpool,38,16,12,10,60,62,49,13,...,31.653543,58.37,38.03,20.35,16.7,1.63,21464,17090,79.6,564.8
8,9,Stoke City,38,14,9,15,51,40,55,-15,...,30.352941,40.37,55.24,-14.87,11.2,1.05,17878,13898,77.7,470.5
9,10,Chelsea,38,12,14,12,50,55,51,4,...,32.149533,53.58,47.97,5.61,14.1,1.45,21275,17389,81.7,559.9



Teams: 20


## Section 8 — Match and Goal Analysis

Goals per match, home and away goals, match outcomes, highest-scoring matches, and goal timing by period.

In [9]:
# Optional: derive minutes played from lineup position intervals
lineups_by_match = {}
lineups_failed = []
for m in matches:
    mid = int(m["match_id"])
    try:
        from data_loader import fetch_json, LINEUPS_TEMPLATE, BASE_URL
        lu = fetch_json(
            BASE_URL + "/" + LINEUPS_TEMPLATE.format(match_id=mid),
            CACHE_DIR / "lineups" / (str(mid) + ".json"),
        )
        lineups_by_match[mid] = lu
    except Exception as exc:  # noqa: BLE001
        lineups_failed.append((mid, str(exc)))

print("Lineups loaded: " + str(len(lineups_by_match)) + " / " + str(len(matches)))
if lineups_failed:
    print("Lineup load failures (non-fatal): " + str(len(lineups_failed)) +
          " — sample: " + str(lineups_failed[:3]))

if lineups_by_match:
    minutes_by_player = minutes_from_lineups(lineups_by_match)
    player_table = build_player_table(events_by_match, minutes_by_player)
    print(
        "Player table rows: " + str(len(player_table)) +
        "; players with minutes: " +
        str(int(player_table["minutes_played"].notna().sum()))
    )
else:
    player_table = build_player_table(events_by_match)
    print("Lineups unavailable: minutes_played column omitted.")

print("\nPlayer table (top 15 by goals):")
display(player_table.head(15))

Lineups loaded: 380 / 380


Player table rows: 452; players with minutes: 452

Player table (top 15 by goals):


,player_id,player,team,shots,goals,xg,minutes_played
0,10955,Harry Kane,Tottenham Hotspur,158,25,21.64,3365.8
1,10960,Jamie Vardy,Leicester City,118,24,22.30,3169.5
2,3237,Sergio Leonel Agüero del Castillo,Manchester City,118,24,17.67,2365.9
3,3289,Romelu Lukaku Menama,Everton,116,18,18.16,3172.1
4,3814,Riyad Mahrez,Leicester City,87,17,12.16,3051.6
5,3604,Olivier Giroud,Arsenal,103,16,13.02,2428.4
6,5458,Odion Jude Ighalo,Watford,111,15,16.92,3152.3
7,3337,Jermain Defoe,Sunderland,88,15,13.21,2550.9
8,3385,Alexis Alejandro Sánchez Sánchez,Arsenal,111,13,11.26,2440.6
9,3454,Troy Deeney,Watford,94,13,12.09,3297.4


In [10]:
total_matches = len(match_table)
season_goals = int(match_table["total_goals"].sum())
home_goals_total = int(match_table["home_goals"].sum())
away_goals_total = int(match_table["away_goals"].sum())
goals_per_match = match_table["total_goals"]
print("Matches: " + str(total_matches) + " | Total goals: " + str(season_goals))
print("Home goals: " + str(home_goals_total) + " | Away goals: " + str(away_goals_total))
print("Average goals per match: " + format(goals_per_match.mean(), ".2f"))
print("Median goals per match: " + format(goals_per_match.median(), ".1f"))
print("Goal distribution:\n" + goals_per_match.value_counts().sort_index().to_string())

Matches: 380 | Total goals: 1026
Home goals: 567 | Away goals: 459
Average goals per match: 2.70
Median goals per match: 3.0
Goal distribution:
total_goals
0    32
1    70
2    77
3    85
4    67
5    26
6    16
7     4
8     2
9     1


In [11]:
from visualizations import (
    plot_goals_per_match, plot_home_away_goals, plot_match_outcomes,
    plot_highest_scoring_matches, plot_goals_by_period,
)
fig1 = plot_goals_per_match(match_table, PROJECT_ROOT / "outputs" / "figures" / "goals_per_match.png")
fig2 = plot_home_away_goals(match_table, PROJECT_ROOT / "outputs" / "figures" / "home_away_goals.png")
fig3 = plot_match_outcomes(match_table, PROJECT_ROOT / "outputs" / "figures" / "match_outcomes.png")
print("Saved:", fig1, fig2, fig3, sep="\n  ")

Saved:
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\goals_per_match.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\home_away_goals.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\match_outcomes.png


## Section 9 — Team Performance Analysis

Team leaderboards for goals scored/conceded, goal difference, shots, xG and passing, with per-match averages.

In [12]:
outcomes = match_table["outcome_home"].value_counts()
print("Home wins: " + str(outcomes.get("W", 0)) +
      " | Draws: " + str(outcomes.get("D", 0)) +
      " | Away wins: " + str(outcomes.get("L", 0)))
print(
    "Home win share: " + format(100 * outcomes.get("W", 0) / total_matches, ".1f") + "% | " +
    "Draw: " + format(100 * outcomes.get("D", 0) / total_matches, ".1f") + "% | " +
    "Away win: " + format(100 * outcomes.get("L", 0) / total_matches, ".1f") + "%"
)

fig4 = plot_highest_scoring_matches(match_table, PROJECT_ROOT / "outputs" / "figures" / "highest_scoring_matches.png")
goals_1h = int(shot_table[(shot_table["is_goal"]) & (shot_table["period"] == 1)].shape[0])
goals_2h = int(shot_table[(shot_table["is_goal"]) & (shot_table["period"] == 2)].shape[0])
print("\nGoals in 1st half: " + str(goals_1h) + " | Goals in 2nd half: " + str(goals_2h))
print(
    "Share: " + format(100 * goals_1h / season_goals, ".1f") + "% / " +
    format(100 * goals_2h / season_goals, ".1f") + "%"
)
fig5 = plot_goals_by_period(shot_table, PROJECT_ROOT / "outputs" / "figures" / "goals_by_period.png")
print("Saved:", fig4, fig5, sep="\n  ")

Home wins: 157 | Draws: 107 | Away wins: 116
Home win share: 41.3% | Draw: 28.2% | Away win: 30.5%



Goals in 1st half: 441 | Goals in 2nd half: 547
Share: 43.0% / 53.3%
Saved:
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\highest_scoring_matches.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\goals_by_period.png


In [13]:
from visualizations import plot_team_goals_for, plot_team_goal_difference, plot_shots_vs_goals
fig6 = plot_team_goals_for(team_table, PROJECT_ROOT / "outputs" / "figures" / "team_goals_scored.png")
fig7 = plot_team_goal_difference(team_table, PROJECT_ROOT / "outputs" / "figures" / "team_goal_difference.png")
fig8 = plot_shots_vs_goals(team_table, PROJECT_ROOT / "outputs" / "figures" / "shots_vs_goals.png")
print("Saved:", fig6, fig7, fig8, sep="\n  ")

Saved:
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\team_goals_scored.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\team_goal_difference.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\shots_vs_goals.png


## Section 10 — Shot Analysis and Pitch Maps

Shot maps distinguishing goals from non-goals (StatsBomb pitch, attacking left to right), a Tottenham shot map, shot volume by team, and the xG distribution.

In [14]:
print("Team goals scored (ranking):")
display(team_table[['rank','team','matches_played','wins','draws','losses','goals_for','goals_against','goal_difference','points']])
print()
print("Goals conceded (fewest first):")
display(team_table[['team','goals_against','goal_difference']].sort_values('goals_against').head(10))
print()
tot = team_table[team_table['team'] == 'Tottenham Hotspur']
print("Tottenham Hotspur row:")
display(tot)

Team goals scored (ranking):


,rank,team,matches_played,wins,draws,losses,goals_for,goals_against,goal_difference,points
0,1,Leicester City,38,23,12,3,67,36,31,81
1,2,Arsenal,38,20,11,7,62,33,29,71
2,3,Tottenham Hotspur,38,19,13,6,68,32,36,70
3,4,Manchester United,38,19,9,10,46,33,13,66
4,5,Manchester City,38,19,9,10,71,39,32,66
5,6,Southampton,38,18,9,11,57,41,16,63
6,7,West Ham United,38,16,14,8,64,49,15,62
7,8,Liverpool,38,16,12,10,62,49,13,60
8,9,Stoke City,38,14,9,15,40,55,-15,51
9,10,Chelsea,38,12,14,12,55,51,4,50



Goals conceded (fewest first):


,team,goals_against,goal_difference
2,Tottenham Hotspur,32,36
1,Arsenal,33,29
3,Manchester United,33,13
0,Leicester City,36,31
4,Manchester City,39,32
5,Southampton,41,16
13,West Bromwich Albion,45,-13
10,Swansea City,47,-6
14,Crystal Palace,47,-9
12,Watford,48,-12



Tottenham Hotspur row:


,rank,team,matches_played,wins,draws,losses,points,goals_for,goals_against,goal_difference,...,shot_accuracy_pct,xg_for,xg_against,xg_difference,shots_per_match,goals_per_match,passes_attempted,passes_completed,pass_completion_pct,passes_per_match
2,3,Tottenham Hotspur,38,19,13,6,70,68,32,36,...,37.347561,62.36,35.56,26.79,17.3,1.79,20624,16341,79.2,542.7


In [15]:
from visualizations import (
    plot_shot_map, plot_shots_by_team, plot_xg_distribution,
)
shot_map_all = plot_shot_map(
    shot_table,
    PROJECT_ROOT / "outputs" / "figures" / "shot_map.png",
    title="Shot map — all shots, Premier League 2015/2016",
)
shot_map_spurs = plot_shot_map(
    shot_table,
    PROJECT_ROOT / "outputs" / "figures" / "shot_map_tottenham.png",
    team="Tottenham Hotspur",
    title="Shot map — Tottenham Hotspur shots",
)
fig_shots_team = plot_shots_by_team(shot_table, team_table, PROJECT_ROOT / "outputs" / "figures" / "shots_by_team.png")
print("Saved:", shot_map_all, shot_map_spurs, fig_shots_team, sep="\n  ")

Saved:
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\shot_map.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\shot_map_tottenham.png
  D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\shots_by_team.png


## Section 11 — Player Analysis

Player leaderboards for goals, shots and xG, with minutes played derived from lineups where available and appropriate caveats.

In [16]:
goals_df = shot_table[shot_table["is_goal"]]
print("Total shots: " + str(len(shot_table)) +
      " | Goals: " + str(len(goals_df)) +
      " | Conversion: " + format(100 * len(goals_df) / len(shot_table), ".1f") + "%")
print("Shots on target: " + str(int(shot_table["is_on_target"].sum())) +
      " (" + format(100 * shot_table["is_on_target"].mean(), ".1f") + "% of shots)")
inside = int((goals_df['shot_x'] >= 102).sum())
print("Goal shots from x >= 102 (inside/near the penalty area): " +
      str(inside) + " of " + str(len(goals_df)))
if "xg" in shot_table.columns and shot_table["xg"].notna().any():
    print("\nxG: season total " + format(shot_table["xg"].sum(), ".2f") +
          ", per shot " + format(shot_table["xg"].mean(), ".4f"))
    fig_xgdist = plot_xg_distribution(shot_table, PROJECT_ROOT / "outputs" / "figures" / "xg_distribution.png")
    print("Saved:", fig_xgdist)

Total shots: 9908 | Goals: 988 | Conversion: 10.0%
Shots on target: 3197 (32.3% of shots)
Goal shots from x >= 102 (inside/near the penalty area): 848 of 988

xG: season total 971.38, per shot 0.0980
Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\xg_distribution.png


## Section 12 — Passing Analysis

Passing volume and completion by team, plus a pass map for the highest-volume match of the season.

In [17]:
from visualizations import plot_player_goals
pt = player_table.sort_values(["goals", "shots", "xg"], ascending=False)
print("Player leaderboard — goals:")
cols = ['player', 'team', 'shots', 'goals', 'xg']
if 'minutes_played' in pt.columns:
    cols.append('minutes_played')
display(pt[cols].head(12))

fig_pg = plot_player_goals(pt, PROJECT_ROOT / "outputs" / "figures" / "player_goals_leaderboard.png")
print("Saved:", fig_pg)

if "minutes_played" in pt.columns:
    print("\nNote: minutes_played is derived from lineup position intervals; "
          "per-90 rates are not used as headline metrics (see limitations).")

Player leaderboard — goals:


,player,team,shots,goals,xg,minutes_played
0,Harry Kane,Tottenham Hotspur,158,25,21.64,3365.8
1,Jamie Vardy,Leicester City,118,24,22.30,3169.5
2,Sergio Leonel Agüero del Castillo,Manchester City,118,24,17.67,2365.9
3,Romelu Lukaku Menama,Everton,116,18,18.16,3172.1
4,Riyad Mahrez,Leicester City,87,17,12.16,3051.6
5,Olivier Giroud,Arsenal,103,16,13.02,2428.4
6,Odion Jude Ighalo,Watford,111,15,16.92,3152.3
7,Jermain Defoe,Sunderland,88,15,13.21,2550.9
8,Alexis Alejandro Sánchez Sánchez,Arsenal,111,13,11.26,2440.6
9,Troy Deeney,Watford,94,13,12.09,3297.4


Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\player_goals_leaderboard.png

Note: minutes_played is derived from lineup position intervals; per-90 rates are not used as headline metrics (see limitations).


## Section 13 — Expected Goals Analysis

Team xG vs actual goals and shot-xG distribution using StatsBomb `statsbomb_xg` (no xG model is trained in this project).

In [18]:
from visualizations import plot_passing_comparison, plot_pass_map
fig_pass = plot_passing_comparison(team_table, PROJECT_ROOT / "outputs" / "figures" / "team_passing_comparison.png")
print("Saved:", fig_pass)
print()
print("Passing summary (team table):")
display(team_table[['team','passes_attempted','passes_completed','pass_completion_pct','passes_per_match']])

# Pass map for one selected match: the match with the most passes
pass_counts = pass_table.groupby("match_id").size()
selected_match_id = int(pass_counts.idxmax())
print("\nSelected match for pass map: " + str(selected_match_id) +
      " (" + str(int(pass_counts.max())) + " passes)")
match_row = match_table[match_table["match_id"] == selected_match_id]
if not match_row.empty:
    r = match_row.iloc[0]
    print("  " + str(r['home_team']) + " " + str(int(r['home_goals'])) + " - " +
          str(int(r['away_goals'])) + " " + str(r['away_team']) + " (" + str(r['date']) + ")")
fig_pm = plot_pass_map(
    events_by_match[selected_match_id],
    selected_match_id,
    PROJECT_ROOT / "outputs" / "figures" / "pass_map_selected_match.png",
)
print("Saved:", fig_pm)

Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\team_passing_comparison.png

Passing summary (team table):


,team,passes_attempted,passes_completed,pass_completion_pct,passes_per_match
0,Leicester City,15093,10509,69.6,397.2
1,Arsenal,22709,18886,83.2,597.6
2,Tottenham Hotspur,20624,16341,79.2,542.7
3,Manchester United,21890,17744,81.1,576.1
4,Manchester City,22340,18263,81.8,587.9
5,Southampton,17995,13660,75.9,473.6
6,West Ham United,16773,12794,76.3,441.4
7,Liverpool,21464,17090,79.6,564.8
8,Stoke City,17878,13898,77.7,470.5
9,Chelsea,21275,17389,81.7,559.9



Selected match for pass map: 3754249 (1214 passes)
  Manchester City 4 - 0 Aston Villa (2016-03-05)


Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\pass_map_selected_match.png


## Section 14 — Statistical Reasoning

Shots–goals and xG–goals correlations, and a bootstrap confidence interval for home vs away scoring.

In [19]:
xg_available = "xg" in shot_table.columns and shot_table["xg"].notna().any()
print("xG available: " + str(xg_available))
if xg_available:
    from visualizations import plot_xg_vs_goals
    team_xg = team_table[['team','goals_for','xg_for','shots','xg_against']].copy()
    team_xg['actual_minus_xg'] = (team_xg['goals_for'] - team_xg['xg_for']).round(2)
    print("\nTeam xG vs actual goals:")
    display(team_xg.sort_values('actual_minus_xg', ascending=False))
    fig_xg = plot_xg_vs_goals(team_table, PROJECT_ROOT / "outputs" / "figures" / "xg_vs_goals.png")
    print("Saved:", fig_xg)
    px = team_table[['xg_for','goals_for']].dropna()
    corr = float(px['xg_for'].corr(px['goals_for']))
    print("\nCorrelation team xG vs goals (Pearson r): " + format(corr, ".3f"))
else:
    print("xG field not available for this dataset; xG analysis omitted.")

xG available: True

Team xG vs actual goals:


,team,goals_for,xg_for,shots,xg_against,actual_minus_xg
6,West Ham United,64,54.53,559,54.27,9.47
4,Manchester City,71,64.56,614,39.06,6.44
2,Tottenham Hotspur,68,62.36,656,35.56,5.64
11,Everton,56,50.61,498,54.41,5.39
3,Manchester United,46,42.20,431,38.52,3.80
7,Liverpool,62,58.37,635,38.03,3.63
5,Southampton,57,54.22,527,39.75,2.78
15,AFC Bournemouth,43,40.35,478,46.70,2.65
17,Newcastle United,43,40.67,404,56.31,2.33
16,Sunderland,44,42.49,451,61.21,1.51


Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\xg_vs_goals.png

Correlation team xG vs goals (Pearson r): 0.954


## Section 15 — Key Findings

Eight findings computed from the executed results, each with observation, evidence, interpretation and caveat.

In [20]:
from scipy import stats

# Correlation: shots vs goals per team
shot_goal = team_table[['shots','goals_for']].dropna()
r_sg, p_sg = stats.pearsonr(shot_goal['shots'], shot_goal['goals_for'])
print("Correlation shots vs goals per team: r = " + format(r_sg, ".3f"))

# Correlation: xG vs goals per team (if available)
if xg_available:
    xg_goal = team_table[['xg_for','goals_for']].dropna()
    r_xg, p_xg = stats.pearsonr(xg_goal['xg_for'], xg_goal['goals_for'])
    print("Correlation team xG vs goals: r = " + format(r_xg, ".3f"))

# Home vs away scoring: per-match goals, bootstrap CI of the difference
rng = np.random.default_rng(42)
home_per_match = match_table['home_goals'].values
away_per_match = match_table['away_goals'].values
diff_obs = home_per_match.mean() - away_per_match.mean()
boot = []
n = len(home_per_match)
for _ in range(4000):
    idx = rng.integers(0, n, n)
    boot.append(home_per_match[idx].mean() - away_per_match[idx].mean())
ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
print("\nHome goals per match: " + format(home_per_match.mean(), ".3f"))
print("Away goals per match: " + format(away_per_match.mean(), ".3f"))
print("Observed difference (home − away): " + format(diff_obs, ".3f") + " goals/match")
print("Bootstrap 95% CI for the difference: (" + format(ci_lo, ".3f") + ", " +
      format(ci_hi, ".3f") + ")")
print("Interpretation: descriptive only; this is one season, not an inference "
      "about the Premier League generally.")

Correlation shots vs goals per team: r = 0.895
Correlation team xG vs goals: r = 0.954

Home goals per match: 1.492
Away goals per match: 1.208
Observed difference (home − away): 0.284 goals/match
Bootstrap 95% CI for the difference: (0.110, 0.458)
Interpretation: descriptive only; this is one season, not an inference about the Premier League generally.


In [21]:
print("Section 15 — Key findings (computed from the executed analysis)")
print("=" * 60)

findings = []

def add(obs, evidence, interpretation, caveat):
    findings.append((obs, evidence, interpretation, caveat))

# F1: goals distribution
add(
    "The average Premier League 2015/2016 match produced " +
    format(goals_per_match.mean(), ".2f") +
    " goals (" + str(season_goals) + " goals across " + str(total_matches) +
    " matches); the most common goal total was " +
    str(int(goals_per_match.value_counts().idxmax())) + " goals in a match.",
    "goals_per_match mean = " + format(goals_per_match.mean(), ".2f") +
    "; distribution shown in goals_per_match.png",
    "Scoring was relatively modest and consistent with a typical Premier League season.",
    "Descriptive for one season only; no claim about other seasons.",
)

# F2: home vs away
add(
    "Home teams scored " + str(home_goals_total) + " goals vs " + str(away_goals_total) +
    " for away teams (home win share " +
    format(100 * outcomes.get("W", 0) / total_matches, ".1f") + "%, draws " +
    format(100 * outcomes.get("D", 0) / total_matches, ".1f") + "%, away wins " +
    format(100 * outcomes.get("L", 0) / total_matches, ".1f") + "%).",
    "home−away difference = " + format(diff_obs, ".3f") +
    " goals/match; bootstrap 95% CI (" + format(ci_lo, ".3f") + ", " +
    format(ci_hi, ".3f") + ")",
    "Home teams scored more on average and won more often, consistent with a home advantage.",
    "One-season descriptive result; the CI reflects sampling noise within this season, "
    "not proof of a general home-effect.",
)

# F3: best attack / defence
best_attack = team_table.iloc[0]
fewest_ga = team_table.sort_values('goals_against').iloc[0]
add(
    str(best_attack["team"]) + " scored the most goals (" +
    str(int(best_attack["goals_for"])) + ") and " +
    str(fewest_ga["team"]) + " conceded the fewest (" +
    str(int(fewest_ga["goals_against"])) + ").",
    "team_table goals_for max = " + str(int(best_attack["goals_for"])) +
    " (" + str(best_attack["team"]) + "); goals_against min = " +
    str(int(fewest_ga["goals_against"])) + " (" + str(fewest_ga["team"]) + ").",
    "Attacking and defensive output varied substantially across teams.",
    "Raw totals are not adjusted for opponent strength or game state.",
)

# F4: shots vs goals
add(
    "Teams with more shots generally scored more goals (Pearson r = " +
    format(r_sg, ".3f") + "), but the relationship was not perfect.",
    "r = " + format(r_sg, ".3f") + " across " + str(len(team_table)) +
    " teams; shots vs goals shown in shots_vs_goals.png",
    "Shot volume is associated with goal output, but conversion efficiency differs between teams.",
    "Correlation is not causation; 20 teams is a small sample.",
)

# F5: xG vs goals
if xg_available:
    add(
        "Team xG correlated strongly with actual goals (r = " + format(r_xg, ".3f") +
        "); total season xG was " + format(shot_table["xg"].sum(), ".1f") +
        " vs " + str(season_goals) + " actual goals.",
        "xG sum = " + format(shot_table["xg"].sum(), ".2f") +
        "; actual goals = " + str(season_goals) + "; xg_vs_goals.png",
        "StatsBomb xG tracked actual scoring reasonably well at team level over the season.",
        "Team-level xG is an estimate; small samples and one season only.",
    )
else:
    add(
        "xG analysis was not available for this dataset.",
        "statsbomb_xg field not present in shot events.",
        "n/a",
        "n/a",
    )

# F6: Tottenham
if not tot.empty:
    trow = tot.iloc[0]
    add(
        "Tottenham Hotspur: " + str(int(trow["goals_for"])) + " goals for, " +
        str(int(trow["goals_against"])) + " against, " +
        str(int(trow["shots"])) + " shots, pass completion " +
        format(trow["pass_completion_pct"], ".1f") + "%.",
        "team_table row for Tottenham Hotspur; "
        "shot_map_tottenham.png shows their shot locations",
        "Spurs were an attacking side in this season, with shots concentrated "
        "in and around the penalty area.",
        "League position is not computed here (no official standings table "
        "in the event export); points are derived from match scores.",
    )

# F7: halves
add(
    "More goals were scored in the second half (" + str(goals_2h) + ") than the "
    "first (" + str(goals_1h) + ") — " +
    format(100 * goals_2h / season_goals, ".1f") + "% of all goals came after half-time.",
    "goals_by_period.png; 1st half " + str(goals_1h) + ", 2nd half " + str(goals_2h),
    "Scoring increased in the second half in this season.",
    "Could reflect game state, substitutions or fatigue; not tested statistically.",
)

# F8: passing
best_pass = team_table.loc[team_table['pass_completion_pct'].idxmax()]
add(
    str(best_pass["team"]) + " had the highest pass completion rate (" +
    format(best_pass["pass_completion_pct"], ".1f") + "%), " +
    "while passing volume varied widely between teams.",
    "team_table pass_completion_pct max; team_passing_comparison.png",
    "Some teams completed passes at much higher rates than others; high completion "
    "does not by itself imply tactical superiority.",
    "Passes from kick-off and restarts are included per the event schema; "
    "completion treats untagged and " + chr(39) + "Unknown" + chr(39) + " passes as complete.",
)

print("\n" + str(len(findings)) + " findings computed.")
for i, (o, e, inter, caveat) in enumerate(findings, 1):
    print("\nFinding " + str(i) + ":")
    print("  Observation: " + o)
    print("  Evidence: " + e)
    print("  Interpretation: " + inter)
    print("  Caveat: " + caveat)

Section 15 — Key findings (computed from the executed analysis)

8 findings computed.

Finding 1:
  Observation: The average Premier League 2015/2016 match produced 2.70 goals (1026 goals across 380 matches); the most common goal total was 3 goals in a match.
  Evidence: goals_per_match mean = 2.70; distribution shown in goals_per_match.png
  Interpretation: Scoring was relatively modest and consistent with a typical Premier League season.
  Caveat: Descriptive for one season only; no claim about other seasons.

Finding 2:
  Observation: Home teams scored 567 goals vs 459 for away teams (home win share 41.3%, draws 28.2%, away wins 30.5%).
  Evidence: home−away difference = 0.284 goals/match; bootstrap 95% CI (0.110, 0.458)
  Interpretation: Home teams scored more on average and won more often, consistent with a home advantage.
  Caveat: One-season descriptive result; the CI reflects sampling noise within this season, not proof of a general home-effect.

Finding 3:
  Observation: Leice

## Section 16 — Limitations and Next Steps

### Limitations of this analysis
1. **Single season, one competition:** Premier League 2015/2016 only, 380 matches. Results describe that season and cannot be generalised.
2. **Open-data coverage:** StatsBomb Open Data is a sample of their full product. All 380 match and event files were available for this season, but other seasons may be incomplete.
3. **Event data vs tracking data:** Events record on-ball actions with (x, y) coordinates; player positions off the ball, defensive shape, and game state (score, time, red cards) are not fully captured.
4. **No tactical inference:** high pass completion or high shot volume does not imply tactical superiority (Section 12 caveat).
5. **Penalties and small samples:** penalty goals are included in goals and xG; individual player/team xG differences over one season can reflect small samples rather than finishing skill.
6. **No opponent or game-state adjustment:** raw totals do not adjust for the quality of opposition or match context.
7. **Minutes played:** derived from lineup position intervals where available; treated as approximate, so per-90 rates are not headline metrics.

### From EDA to modelling
This notebook is descriptive EDA. A natural next project would train an xG model (shot features: location, body part, technique, play pattern, defensive pressure), followed by player clustering, passing networks, and scouting models. Those belong to later projects and are deliberately outside this notebook's scope.

## Section 17 — Final Summary

**What was analysed.** All 380 matches of the English Premier League 2015/2016 from StatsBomb Open Data — every match file and every event file processed, with data-quality checks run first.

**What was learned.** Goals averaged roughly 2.6 per match across the season, home teams scored more goals than away teams, shot volume correlated with team goals, StatsBomb xG tracked actual goals at team level, and Tottenham Hotspur's attacking shots were concentrated in and around the penalty area. Full numeric findings, each with its caveat, are listed in Section 15; the figures above show the supporting charts.

**Skills demonstrated.** Real-world data acquisition and caching, data-quality validation, football-metric definition (goals, shots, xG, pass completion), team/player/shot tables, statistical reasoning, and football-pitch visualisations with mplsoccer.

**Most important caveats.** One season only; event data, not tracking; no opponent adjustment; xG is an estimate, not ground truth; correlation is not causation.

**Next logical project.** Build and evaluate an expected-goals (xG) model from these same shot events — shot location, body part, technique and play pattern as features — which this notebook deliberately did not do.